In [1]:
!pip install ultralytics opencv-python deep-sort-realtime pandas

In [5]:
# Import Libraries
import cv2
import numpy as np
import pandas as pd
from datetime import datetime
from ultralytics import YOLO # YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort # DeepSORT

import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as keras_preprocess
from tensorflow.keras.preprocessing.image import img_to_array

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [6]:
def preprocess_frame(frame, resize_dim=(640, 480), denoise=False):
    """
    Preprocess the input frame by resizing, denoising, etc.

    Args:
        frame (np.ndarray): Original video frame from OpenCV.
        resize_dim (tuple): Desired (width, height) to resize frame.
        denoise (bool): Flag to apply a denoising filter if needed.

    Returns:
        preprocessed_frame (np.ndarray): The preprocessed frame.
    """
    # 1. Resize the frame
    resized_frame = cv2.resize(frame, resize_dim)

    # 2. Denoising - remove or reduce noise
    if denoise:
        resized_frame = cv2.fastNlMeansDenoisingColored(resized_frame, None, 10, 10, 7, 21)

    # Additional preprocessing steps could include:
    # - Normalizing pixel values
    # - Converting color spaces if needed (e.g., HSV)
    # - Gamma correction, etc.

    return resized_frame

In [7]:
def filter_vehicle_detections(yolo_result_boxes, class_names, confidence_threshold=0.3):
    """
    Filter YOLO detections to keep only vehicle classes, returning bounding boxes in the format
    [x1, y1, x2, y2, confidence], along with their class IDs.

    Args:
        yolo_result_boxes (list): YOLO result.boxes from ultralytics model.
        class_names (dict): Mapping from class ID to class name (e.g. {2:'car',3:'motorcycle',...})
        confidence_threshold (float): Minimum confidence for keeping a detection.

    Return:
        a list of detection data for vehicles:
            Each element: ([x1, y1, x2, y2, conf], cls_id)
    """
    detections = []
    for box in yolo_result_boxes:
        cls_id = int(box.cls[0])  # Class ID
        conf = float(box.conf[0]) # Confidence
        if conf < confidence_threshold:
            continue
        
        # Only keep detections if they are in our vehicle classes
        if cls_id in class_names:
            x1, y1, x2, y2 = box.xyxy[0]
            detections.append(([int(x1), int(y1), int(x2), int(y2), conf], cls_id))
    return detections

In [8]:
def main(video_source=0,
         resize_dim=(640, 480),
         denoise=False,
         line_position=250,
         output_csv="vehicle_counts.csv",
         output_video="annotated_output.avi",
         detection_interval=3,
         confidence_threshold=0.5):
    """
    Main function that performs:
        1) Video capture
        2) Frame preprocessing
        3) YOLO (Ultralytics) detection
        4) DeepSORT tracking
        5) Vehicle counting via line crossing
        6) CSV logging
        7) Video writing

    Args:
        video_source (str/int): Path to video or webcam index.
        resize_dim (tuple): (width, height) for resizing frames.
        denoise (bool): Apply denoising or not.
        line_position (int): y-coordinate of the horizontal line for counting.
        output_csv (str): Path to save crossing log.
        output_video (str): Path to save the annotated video.
        confidence_threshold (float): Min confidence for detections.
    """

    # -------------------------
    # 0. Class name definitions
    #    (COCO IDs for vehicles)
    # -------------------------
    class_names = {
        1: 'bicycle',
        2: 'car',
        3: 'motorcycle',
        5: 'bus',
        7: 'truck'
    }
    
    # 5 class from classification model
    custom_classes = [
        "Commercial Vehicles",
        "High-End Vehicles",
        "Low-End Vehicles",
        "Mid-Range Vehicles",
        "Motorcycle"
    ]

    MIN_W = 10
    MIN_H = 10
    THRESHOLD = 0.6

    # -------------------------
    # 1. Initialize models (YOLO for detection, MobileNetV3 for classification)
    # -------------------------
    model = YOLO("yolo11n.pt")  
    classifier_model = tf.keras.models.load_model("mobilenetv3_original.keras")

    # -------------------------
    # 2. Initialize Video Capture
    # -------------------------
    cap = cv2.VideoCapture(video_source)
    if not cap.isOpened():
        print(f"Error: Cannot open video source {video_source}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    out_width, out_height = resize_dim

    # -------------------------
    #      Video Writer
    # -------------------------
    fourcc = cv2.VideoWriter_fourcc(*"XVID")
    out_writer = cv2.VideoWriter(output_video, fourcc, fps, (out_width, out_height))

    # -------------------------
    # 3. Initialize DeepSORT
    # -------------------------
    deepsort = DeepSort(
        max_age=15,
        n_init=2,
        nms_max_overlap=1.0,
        max_cosine_distance=0.2,
        nn_budget=25,
        embedder="mobilenet",
        embedder_gpu=True      # Enable GPU for faster embedding
    )

    # -------------------------
    # 4. CSV Logging
    # -------------------------
    # Remove existing CSV if it exists
    if os.path.exists(output_csv):
        os.remove(output_csv)

    # Prepare a DataFrame to log crossing events
    df_log = pd.DataFrame(columns=["timestamp", "track_id", "vehicle_type", "direction"])

    # track how many vehicles cross from up->down and down->up
    total_up_down = 0
    total_down_up = 0

    # For each track, store its last center Y and whether we've already counted a crossing
    # track_memory[track_id] = {"last_y": y, "counted": False}
    track_memory = {}
    track_classes = {} # dict for track_id -> "High-End", etc. 

    paused = False

     # -------------------------
    # 5. Pipeline
    # -------------------------
    frame_idx = 0
    while True:
        try:
            if not paused:
                ret, frame = cap.read()
                if not ret:
                    print("End of video stream or cannot fetch the frame.")
                    break
                if frame is None or frame.size == 0:
                    print("Empty frame, skipping...")
                    continue
                frame_idx += 1
    
                # 5.1 Preprocess frame
                preprocessed_frame = preprocess_frame(frame, resize_dim, denoise)

                if frame_idx % detection_interval == 0: # frame skipping
                    # 5.2 YOLO Inference
                    # Each result can have multiple boxes (detections)
                    yolo_results = model(preprocessed_frame, verbose=False) # cuda can be used if available
        
                    # aggregate all detections in a list
                    detections_for_tracker = []
                    for result in yolo_results:
                        filtered_dets = filter_vehicle_detections(
                            result.boxes,
                            class_names,
                            confidence_threshold
                        )
                        # filtered_dets is [([x1, y1, x2, y2, conf], cls_id), ...]
                        detections_for_tracker.extend(filtered_dets)
        
                    # 5.3 Prepare detection data for DeepSORT
                    bbs = []
                    for box_data, cls_id in detections_for_tracker:
                        x1, y1, x2, y2, conf = box_data
                    
                        # Convert from (x1, y1, x2, y2) -> (x, y, w, h)
                        w = x2 - x1
                        h = y2 - y1
                    
                        # Each detection for DeepSORT: [[x, y, w, h], conf, class_id]
                        bbs.append([[x1, y1, w, h], conf, cls_id])
                        
                    # 5.4 Update DeepSORT
                    tracks = deepsort.update_tracks(bbs, frame=preprocessed_frame)
                    # tracks: list of track objects
    
                    new_crops, new_ids = [], []
                    for track in tracks:
                        if not track.is_confirmed(): 
                            continue
                        tid = track.track_id
                        # only collect if haven’t classified this track before
                        if tid not in track_classes:
                            x1,y1,x2,y2 = map(int, track.to_ltrb())
                            crop = frame[y1:y2, x1:x2]
                            # only if reasonably large to avoid partial bodies
                            if crop.shape[0] > MIN_H and crop.shape[1] > MIN_W:
                                new_crops.append(crop)
                                new_ids.append(tid)
                
                    valid_pairs = []
                    for c, tid in zip(new_crops, new_ids):
                        h, w = c.shape[:2]
                        if h > 0 and w > 0:
                            valid_pairs.append((c, tid))
                    
                    if valid_pairs:
                        crops, tids = zip(*valid_pairs)
                    
                        # Build the batch safely
                        batch_list = []
                        for c in crops:
                            # convert to RGB, resize, preprocess
                            rgb   = cv2.cvtColor(c, cv2.COLOR_BGR2RGB)
                            small = cv2.resize(rgb, (224,224))
                            arr   = img_to_array(small)
                            arr   = keras_preprocess(arr)
                            batch_list.append(arr)
                    
                        batch = np.stack(batch_list, axis=0)  # shape (N,224,224,3)
                    
                        # batched inference
                        probs  = classifier_model.predict_on_batch(batch)   # shape (N,5)
                        labels = probs.argmax(axis=1)                       # shape (N,)
                        confs  = probs.max(axis=1)                          # shape (N,)                       
                        
                        for tid, lab, conf in zip(tids, labels, confs):
                            if conf >= THRESHOLD:
                                track_classes[tid] = custom_classes[lab]
                                print(track_classes)

                else:
                    # ---- NO DETECTION: just advance tracker ----
                    tracks = deepsort.update_tracks([], frame=preprocessed_frame)
                
                # 5.5 Counting Logic
                # For each confirmed track, get ID, bounding box, class, etc.
                for track in tracks:
                    if not track.is_confirmed(): # DeepSORT tracks can be 'confirmed' or 'tentative'
                        continue

                    # 1) get bbox
                    x1, y1, x2, y2 = map(int, track.to_ltrb()) # track.to_ltrb() gives (left, top, right, bottom)

                    track_id = track.track_id
                    cls_label = track_classes.get(track_id, "Unclassified")
    
                    # Compute center
                    cx = int((x1 + x2) / 2)
                    cy = int((y1 + y2) / 2)
    
                    # Retrieve memory
                    mem = track_memory.get(track_id, {"last_y": cy, "counted": False})
                    last_y = mem["last_y"]
                    counted = mem["counted"]
    
                    # Check crossing from above -> below
                    if not counted and last_y < line_position <= cy:
                        total_up_down += 1
                        counted = True
                        event_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                        df_log.loc[len(df_log)] = [event_time, track_id, cls_label, "up->down"]
    
                    # Check crossing from below -> above
                    elif not counted and last_y > line_position >= cy:
                        total_down_up += 1
                        counted = True
                        event_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                        df_log.loc[len(df_log)] = [event_time, track_id, cls_label, "down->up"]
    
                    # Update memory
                    track_memory[track_id] = {"last_y": cy, "counted": counted}
    
                    # 5.6 Draw bounding box and ID
                    color = (0, 255, 0)
                    cv2.rectangle(preprocessed_frame, (x1, y1), (x2, y2), color, 2)
                    label_text = f"ID:{track_id} {cls_label}"
                    cv2.putText(preprocessed_frame, label_text, (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
                # 5.7 Draw the counting line & counting info
                cv2.line(preprocessed_frame, (0, line_position),
                         (out_width, line_position), (0, 0, 255), 2)
                info_text = f"Up->Down: {total_up_down} | Down->Up: {total_down_up}"
                cv2.putText(preprocessed_frame, info_text, (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
                # 5.8 Show frame & write to output
                cv2.imshow("YOLO + DeepSORT Tracking & Counting", preprocessed_frame)
                out_writer.write(preprocessed_frame)
        except Exception as e:
            print(f"Error processing frame: {e}")
            break

        # -------------------------
        # Check keyboard events
        # -------------------------
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            # Quit
            break
        elif key == ord('p'):
            # Toggle pause
            paused = not paused
            if paused:
                print("Paused. Press 'c' to continue.")
        elif key == ord('c'):
            # Continue
            paused = False
            print("Continuing...")

    # -------------------------
    # Cleanup
    # -------------------------
    cap.release()
    out_writer.release()
    cv2.destroyAllWindows()

    # Save CSV log
    df_log.to_csv(output_csv, index=False)
    print(f"[INFO] Counting log saved to: {output_csv}")
    print(f"[INFO] Processed video saved to: {output_video}")

In [9]:
if __name__ == "__main__":
    """
       - Provide a video file, or an integer for webcam index.
       - Provide the line_position for counting.
       - Adjust confidence_threshold for YOLO.

    main(video_source="traffic_video.mp4",
         resize_dim=(640, 480),
         denoise=False,
         line_position=250,
         output_csv="vehicle_counts.csv",
         output_video="annotated_output.avi",
         confidence_threshold=0.3)
    """
    
    main(
        video_source="cam dataset/Krubong Junctions/Cam2/8-8.15AM/video.mp4",
        resize_dim=(640, 480),
        denoise=False,
        line_position=250,
        output_csv="vehicle_counts.csv",
        output_video="counting and tracking.avi",
        detection_interval=3,
        confidence_threshold=0.5
    )

[INFO] Counting log saved to: vehicle_counts.csv
[INFO] Processed video saved to: counting and tracking.avi
